# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wajiha-Waqar/FlyRankInternship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

#My Rule

The purpose of this baseline is to prioritize pages that may deserve review before training any machine learning model.

Since this warehouse slice does not contain a content freshness feature, I based my rule on the signals that are available.

A page receives a higher priority when it:

has strong search visibility (higher impressions)
has a lower-than-expected CTR relative to its search position

These pages may represent opportunities where users see the page frequently but click less than expected.

The baseline is intentionally simple and transparent so that every recommendation can be explained before introducing machine learning.
Because this warehouse slice does not include a freshness signal, the baseline uses only visibility and CTR. Freshness will be incorporated in future iterations when available.

Reason Codes
| Reason Code             | Meaning                                                     |
| ----------------------- | ----------------------------------------------------------- |
| high_visibility_low_ctr | High impressions and below-threshold CTR. Highest priority. |
| high_visibility         | Strong visibility but acceptable CTR. Monitor.              |
| low_ctr                 | Low CTR despite lower visibility. Review if needed.         |


Monitor

Pages satisfying the rule should be reviewed by the SEO team to determine whether a content refresh or CTR improvement is appropriate.



#Loading March Data

In [3]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

con.sql(f"""
CREATE SECRET hf_secret (
TYPE HUGGINGFACE,
TOKEN '{HF_TOKEN}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [4]:
con.execute("""
CREATE TABLE fact_content_daily_performance AS
SELECT *
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
con.execute("SHOW TABLES").fetchdf()

,name
0,fact_content_daily_performance


In [6]:
march_df = con.execute("""
SELECT *
FROM fact_content_daily_performance
WHERE month='2026-03'
AND ga4_data_available IS TRUE
""").df()

march_df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_65de48885f4ef01b,content_09be8cc7fcb222af,True,True,False,True,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-03
1,2026-03-01,client_65de48885f4ef01b,content_851afac9fe13612e,True,True,False,True,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-03
2,2026-03-01,client_65de48885f4ef01b,content_cee6c6fc8c51af14,True,True,False,True,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-03
3,2026-03-01,client_65de48885f4ef01b,content_5e120e972f11f833,True,True,False,True,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-03
4,2026-03-01,client_65de48885f4ef01b,content_16a7291bb6ecaebe,True,True,False,True,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-03


In [7]:
march_df.columns.to_list()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

#Create CTR

In [8]:
march_df["ctr"] = np.where(
    march_df["gsc_impressions"] > 0,
   march_df["gsc_clicks"] / march_df["gsc_impressions"],
    np.nan
)

#Signal 1
High Visibility

In [9]:
march_df["impression_bucket"] = pd.cut(
    march_df["gsc_impressions"],
    bins=[0,100,500,1000,5000,march_df["gsc_impressions"].max()+1],
    labels=[
        "0-100",
        "101-500",
        "501-1000",
        "1001-5000",
        "5000+"
    ]
)

signal1 = (
    march_df
    .groupby("impression_bucket")
    .agg(
        avg_clicks=("gsc_clicks","mean"),
        avg_ctr=("ctr","mean"),
        n=("gsc_impressions","size")
    )
)

signal1

/tmp/ipykernel_25092/3628833557.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("impression_bucket")


,avg_clicks,avg_ctr,n
impression_bucket,,,
0-100,0.421767,0.021602,184595
101-500,1.173231,0.005539,138232
501-1000,2.518783,0.003692,26726
1001-5000,5.103700,0.003033,14407
5000+,37.012920,0.003797,387


#Signal 1 Verdict:

CONFIRMED

Higher impression buckets consistently receive more clicks, indicating that impressions are a useful measure of search visibility. This supports using visibility as one component of the baseline prioritization rule.

#Signal 2

CTR vs Position

In [10]:
march_df["position_bucket"] = pd.cut(
    march_df["gsc_avg_position"],
    bins=[0,3,10,20,50,100],
    labels=[
        "Top 3",
        "4-10",
        "11-20",
        "21-50",
        "51-100"
    ]
)

signal2 = (
    march_df
    .groupby("position_bucket")
    .agg(
        avg_ctr=("ctr","mean"),
        avg_impressions=("gsc_impressions","mean"),
        n=("ctr","size")
    )
)

signal2

/tmp/ipykernel_25092/3823799219.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("position_bucket")


,avg_ctr,avg_impressions,n
position_bucket,,,
Top 3,0.024980,248.515223,43847
4-10,0.015466,235.268465,141322
11-20,0.010053,170.590544,75610
21-50,0.005473,292.014338,94920
51-100,0.009033,106.726488,5210


#Signal 2 Verdict:
MIXED

CTR generally decreases as search position becomes worse, matching the expected relationship for most buckets. However, the 51–100 bucket has a slightly higher average CTR than the 21–50 bucket, so the trend is not perfectly consistent. The signal is still useful but should not be relied upon in isolation.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

#High visibility

In [11]:
march_df["high_visibility"] = (
    march_df["gsc_impressions"] >= 500
).astype(int)

#CTR

In [12]:
ctr_threshold = march_df["ctr"].median()

march_df["low_ctr"] = (
    march_df["ctr"] < ctr_threshold
).astype(int)

#Building the Baseline Score

This rule uses only information that would have been available at decision time.

The score is intentionally transparent rather than optimized.

Each page receives one point when it exceeds the visibility threshold.

The resulting ranked queue is written to:

work/outputs/baseline_action_score.csv

This baseline will become the benchmark that future machine learning models must outperform.

In [13]:
march_df["score"] = (
    (march_df["gsc_impressions"] >= 500).astype(int)
    +
    (march_df["ctr"] < 0.01).astype(int)
)

#Reason code

In [14]:
march_df["reason_code"] = np.select(
    [
        march_df["score"] == 2,
        march_df["gsc_impressions"] >= 500,
        march_df["ctr"] < 0.01,
    ],
    [
        "high_visibility_low_ctr",
        "high_visibility",
        "low_ctr",
    ],
    default="none",
)

#Action label
Refresh Review and Monitor

In [15]:
march_df["action_label"] = np.where(
    march_df["score"] >= 2,
    "Refresh Review",
    "Monitor"
)

#Rank

In [16]:
baseline = (
    march_df
    .sort_values(
        by="score",
        ascending=False
    )
)

#Save CSV

In [17]:
import os

os.makedirs(
    "work/outputs",
    exist_ok=True
)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

#Print top rows

In [18]:
baseline[
    [
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "score",
        "reason_code",
        "action_label"
    ]
].head(20)

,content_hash_id,gsc_impressions,gsc_clicks,ctr,score,reason_code,action_label
293769,content_39fe6d6bb78e11a5,586,0,0.000000,2,high_visibility_low_ctr,Refresh Review
293768,content_227d9704d7286471,776,6,0.007732,2,high_visibility_low_ctr,Refresh Review
250678,content_097459d155cccb26,824,1,0.001214,2,high_visibility_low_ctr,Refresh Review
293764,content_7dc004367303a7cf,627,3,0.004785,2,high_visibility_low_ctr,Refresh Review
34463,content_14df7b049d1d6467,1309,3,0.002292,2,high_visibility_low_ctr,Refresh Review
172493,content_3240e96b9a0da4f7,661,2,0.003026,2,high_visibility_low_ctr,Refresh Review
293790,content_2eecb7dc09096daf,539,1,0.001855,2,high_visibility_low_ctr,Refresh Review
293787,content_6f4030445fe83b5a,627,5,0.007974,2,high_visibility_low_ctr,Refresh Review
68572,content_ad81eee3c0435e0b,561,2,0.003565,2,high_visibility_low_ctr,Refresh Review
193925,content_d69718c64f0f46a9,1274,2,0.001570,2,high_visibility_low_ctr,Refresh Review


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

#Top-20 Review

The purpose of reviewing the highest-ranked pages is to check whether the rule behaves reasonably before trusting it.

Each recommendation was inspected manually using the available warehouse signals.
| Rank | Action  | Why it is here                              | Confidence | What would make it wrong                                                                 |
| ---- | ------- | ------------------------------------------- | ---------- | ---------------------------------------------------------------------------------------- |
| 1    | Monitor | High search visibility (746 impressions).   | **Medium** | High impressions alone may not justify a refresh if the page is already performing well. |
| 2    | Monitor | 624 impressions with very low CTR.          | **High**   | Low CTR may be caused by search intent rather than poor content.                         |
| 3    | Monitor | Strong visibility but weak CTR.             | **High**   | The page could already satisfy user intent despite low clicks.                           |
| 4    | Monitor | More than 1,200 impressions.                | **Medium** | A good CTR may indicate no refresh is needed.                                            |
| 5    | Monitor | Nearly 2,000 impressions.                   | **Medium** | High visibility alone does not indicate declining quality.                               |
| 6    | Monitor | Very high visibility (4,166 impressions).   | **Medium** | Traffic volume could simply reflect a popular evergreen page.                            |
| 7    | Monitor | 920 impressions with zero CTR.              | **High**   | Missing or incomplete tracking could explain the result.                                 |
| 8    | Monitor | Good visibility with weak CTR.              | **High**   | SERP competition may explain poor CTR instead of page quality.                           |
| 9    | Monitor | 565 impressions and zero CTR.               | **High**   | The page may target informational searches where clicks are naturally low.               |
| 10   | Monitor | 733 impressions.                            | **Medium** | CTR alone does not prove content should be refreshed.                                    |
| 11   | Monitor | High visibility with relatively strong CTR. | **Low**    | This may actually be a false positive because CTR is already healthy.                    |
| 12   | Monitor | 622 impressions.                            | **Medium** | Ranking changes outside the observation window could explain performance.                |
| 13   | Monitor | 595 impressions.                            | **Medium** | Visibility threshold alone may over-prioritize this page.                                |
| 14   | Monitor | 650 impressions.                            | **Low**    | Performance may already be acceptable for its search position.                           |
| 15   | Monitor | 616 impressions.                            | **Low**    | Competition in search results may be the true issue.                                     |
| 16   | Monitor | 534 impressions.                            | **Low**    | Traffic may fluctuate naturally over time.                                               |
| 17   | Monitor | 971 impressions.                            | **Medium** | Low CTR may reflect misleading titles rather than stale content.                         |
| 18   | Monitor | 1,182 impressions.                          | **Medium** | Search demand could explain the high impressions without needing a refresh.              |
| 19   | Monitor | 512 impressions.                            | **Low**    | The page may already meet business goals despite low CTR.                                |
| 20   | Monitor | 1,262 impressions.                          | **Medium** | High impressions do not always imply an optimization opportunity.                        |
                     

In [19]:
top20 = baseline.head(20)

top20[
[
"content_hash_id",
"score",
"reason_code",
"action_label",
"gsc_impressions",
"ctr",
"gsc_avg_position"
]
]

,content_hash_id,score,reason_code,action_label,gsc_impressions,ctr,gsc_avg_position
293769,content_39fe6d6bb78e11a5,2,high_visibility_low_ctr,Refresh Review,586,0.000000,7.994881
293768,content_227d9704d7286471,2,high_visibility_low_ctr,Refresh Review,776,0.007732,14.007732
250678,content_097459d155cccb26,2,high_visibility_low_ctr,Refresh Review,824,0.001214,19.947816
293764,content_7dc004367303a7cf,2,high_visibility_low_ctr,Refresh Review,627,0.004785,6.287081
34463,content_14df7b049d1d6467,2,high_visibility_low_ctr,Refresh Review,1309,0.002292,30.879297
172493,content_3240e96b9a0da4f7,2,high_visibility_low_ctr,Refresh Review,661,0.003026,29.493192
293790,content_2eecb7dc09096daf,2,high_visibility_low_ctr,Refresh Review,539,0.001855,25.671614
293787,content_6f4030445fe83b5a,2,high_visibility_low_ctr,Refresh Review,627,0.007974,19.655502
68572,content_ad81eee3c0435e0b,2,high_visibility_low_ctr,Refresh Review,561,0.003565,4.481283
193925,content_d69718c64f0f46a9,2,high_visibility_low_ctr,Refresh Review,1274,0.001570,5.540031


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

#Weak Picks and Leakage Check
Weak Picks

The review identified several pages that were likely false positives.

The current rule primarily rewards high visibility because a freshness signal was not available in the warehouse slice.

As a result, some highly visible pages with acceptable performance were still ranked near the top.

Future versions should incorporate additional signals such as:

content freshness
historical performance trends
position changes over time
engagement metrics

These additional signals should improve ranking quality.

#Leakage Check

The baseline intentionally avoids future information.

The rule uses only:

GSC impressions
GSC clicks (used only to compute current CTR)
Average search position

No future performance labels, outcome variables, or label-derived columns were used.

Therefore, the baseline does not leak future information.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.